In [ ]:
from sentence_transformers import SentenceTransformer

st_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=DEVICE)

def minilm_probs(df):
    p_emb = st_model.encode(df["prompt_clean"].tolist(), normalize_embeddings=True,
                            convert_to_numpy=True, batch_size=64)
    sims = np.zeros((len(df), 5), dtype=np.float32)
    for i, o in enumerate(OPTS):
        o_emb = st_model.encode(df[o].tolist(), normalize_embeddings=True,
                                convert_to_numpy=True, batch_size=64)
        sims[:, i] = (p_emb * o_emb).sum(axis=1)     # cosine (already normalised)
    e = np.exp((sims - sims.max(axis=1, keepdims=True)) / 0.1)   # temperature softmax
    return e / e.sum(axis=1, keepdims=True)